In [4]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    free, total = torch.cuda.mem_get_info()
    print(f"Free VRAM: {free / 1024**3:.2f} GB")
    print(f"Total VRAM: {total / 1024**3:.2f} GB")
    print("CUDA:", torch.version.cuda)

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
Free VRAM: 14.46 GB
Total VRAM: 14.56 GB
CUDA: 12.8


In [6]:
import sys
print("Python:", sys.version)
print("Executable:", sys.executable)

Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
Executable: /usr/bin/python3


In [7]:
%pip install -U uv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.4/20.4 MB 93.2 MB/s eta 0:00:00:00:0100:01


In [9]:
!uv pip install --python /content/vllm-env/bin/python vllm --torch-backend=auto

Using Python 3.12.3 environment at: vllm-env
Resolved 196 packages in 5.81s                                       
Prepared 196 packages in 1m 13s                                          
Installed 196 packages in 1.78s29.7                         
 + agent-detector==2.0.0
 + aiohappyeyeballs==2.7.1
 + aiohttp==3.14.3
 + aiosignal==1.4.0
 + annotated-doc==0.0.5
 + annotated-types==0.8.0
 + anthropic==1.7.0
 + anyio==4.15.1
 + apache-tvm-ffi==0.1.11
 + astor==0.8.1
 + attrs==26.1.0
 + blake3==1.0.9
 + cachetools==7.2.0
 + cbor2==6.1.4
 + certifi==2026.7.22
 + cffi==2.1.1
 + charset-normalizer==3.5.1
 + click==8.5.0
 + cloudpickle==3.1.2
 + compressed-tensors==0.17.0
 + cryptography==50.0.1
 + cuda-bindings==13.4.2
 + cuda-core==1.2.0
 + cuda-pathfinder==1.8.2
 + cuda-python==13.4.1
 + cuda-tile==1.6.0
 + cuda-toolkit==13.2.1
 + depyf==0.20.0
 + detect-installer==0.2.1
 + dill==0.4.1
 + dnspython==2.8.0
 + docstring-parser==0.18.0
 + einops==0.8.2
 + email-validator==2.3.0
 + fastapi==0

In [10]:
! /content/vllm-env/bin/python -c "import vllm, torch; print('vLLM:', vllm.__version__); print('PyTorch:', torch.__version__); print('CUDA:', torch.version.cuda); print('CUDA available:', torch.cuda.is_available()); print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')"

vLLM: 0.29.0
PyTorch: 2.13.0+cu132
CUDA: 13.2
CUDA available: True
GPU: Tesla T4


In [13]:
! /content/vllm-env/bin/python -c "from huggingface_hub import snapshot_download; path=snapshot_download('sanskar003/Qwen3.5-9B-AWQ'); print('Downloaded to:', path)"


Downloaded to: /root/.cache/huggingface/hub/models--sanskar003--Qwen3.5-9B-AWQ/snapshots/acf12fac67b9591c99f97f6716a9d0e7c20bbff5




In [17]:
import subprocess

code = r'''
from vllm import LLM

print("Loading Qwen3.5-9B quantized model...", flush=True)

model = LLM(
    model="/root/.cache/huggingface/hub/models--sanskar003--Qwen3.5-9B-AWQ/snapshots/acf12fac67b9591c99f97f6716a9d0e7c20bbff5",
    dtype="float16",
    max_model_len=2048,
    max_num_seqs=8,
    gpu_memory_utilization=0.80,
)

print("MODEL LOAD: SUCCESS", flush=True)
'''

result = subprocess.run(
    ["/content/vllm-env/bin/python", "-u", "-c", code],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

print(result.stdout)
print("\nProcess exit code:", result.returncode)

Loading Qwen3.5-9B quantized model...
INFO 09-21 03:42:05 [api_utils.py:286] non-default args: {'dtype': 'float16', 'max_model_len': 2048, 'gpu_memory_utilization': 0.8, 'max_num_seqs': 8, 'disable_log_stats': True, 'model': '/root/.cache/huggingface/hub/models--sanskar003--Qwen3.5-9B-AWQ/snapshots/acf12fac67b9591c99f97f6716a9d0e7c20bbff5'}
INFO 09-21 03:42:05 [model.py:684] Resolved architecture: Qwen3_5ForConditionalGeneration
WARNING 09-21 03:42:05 [model.py:2355] Casting torch.bfloat16 to torch.float16.
INFO 09-21 03:42:05 [model.py:2021] Using max model len 2048
WARNING 09-21 03:42:05 [model.py:981] Model does not support mm_device_do_normalize, forcing mm_device_do_normalize = False.
INFO 09-21 03:42:12 [scheduler.py:277] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 09-21 03:42:12 [config.py:625] Mamba cache mode is set to 'align' for Qwen3_5ForConditionalGeneration by default when prefix caching is enabled
INFO 09-21 03:42:12 [kernel.py:369] Final IR op prio

In [1]:
! /content/vllm-env/bin/vllm serve \
    /root/.cache/huggingface/hub/models--sanskar003--Qwen3.5-9B-AWQ/snapshots/acf12fac67b9591c99f97f6716a9d0e7c20bbff5 \
    --dtype float16 \
    --max-model-len 2048 \
    --max-num-seqs 8 \
    --gpu-memory-utilization 0.80 \
    --host 0.0.0.0 \
    --port 8000 \
    --served-model-name qwen3.5-9b

(APIServer pid=23390) INFO 09-21 04:23:28 [api_utils.py:347] 
(APIServer pid=23390) INFO 09-21 04:23:28 [api_utils.py:347]        █     █     █▄   ▄█
(APIServer pid=23390) INFO 09-21 04:23:28 [api_utils.py:347]  ▄▄ ▄█ █     █     █ ▀▄▀ █  version 0.29.0
(APIServer pid=23390) INFO 09-21 04:23:28 [api_utils.py:347]   █▄█▀ █     █     █     █  model   /root/.cache/huggingface/hub/models--sanskar003--Qwen3.5-9B-AWQ/snapshots/acf12fac67b9591c99f97f6716a9d0e7c20bbff5
(APIServer pid=23390) INFO 09-21 04:23:28 [api_utils.py:347]    ▀▀  ▀▀▀▀▀ ▀▀▀▀▀ ▀     ▀
(APIServer pid=23390) INFO 09-21 04:23:28 [api_utils.py:347] 
(APIServer pid=23390) INFO 09-21 04:23:28 [api_utils.py:286] non-default args: {'model_tag': '/root/.cache/huggingface/hub/models--sanskar003--Qwen3.5-9B-AWQ/snapshots/acf12fac67b9591c99f97f6716a9d0e7c20bbff5', 'host': '0.0.0.0', 'model': '/root/.cache/huggingface/hub/models--sanskar003--Qwen3.5-9B-AWQ/snapshots/acf12fac67b9591c99f97f6716a9d0e7c20bbff5', 'dtype': 'float16', 'max_mo

In [4]:
import subprocess

print("=== vLLM process ===")
print(subprocess.run(
    ["bash", "-c", "ps aux | grep '[v]llm'"],
    capture_output=True,
    text=True
).stdout)

print("\n=== Port 8000 ===")
print(subprocess.run(
    ["bash", "-c", "ss -ltnp | grep ':8000' || true"],
    capture_output=True,
    text=True
).stdout)

=== vLLM process ===


=== Port 8000 ===



In [5]:
import subprocess

result = subprocess.run(
    ["bash", "-c", "dmesg | tail -30"],
    capture_output=True,
    text=True
)

print(result.stdout)

[   16.312228] NVRM: loading NVIDIA UNIX Open Kernel Module for x86_64  580.82.07  Release Build  (builder@28e54e79972f)  Thu Apr 30 18:50:30 UTC 2026
[   16.334878] LoadPin: kernel-module pinning-excluded obj="/usr/local/nvidia/drivers/nvidia-uvm.ko" pid=1323 cmdline="insmod /usr/local/nvidia/drivers/nvidia-uvm.ko"
[   16.656971] LoadPin: kernel-module pinning-excluded obj="/usr/local/nvidia/drivers/nvidia-modeset.ko" pid=1329 cmdline="insmod /usr/local/nvidia/drivers/nvidia-modeset.ko"
[   16.815050] nvidia-modeset: Loading NVIDIA UNIX Open Kernel Mode Setting Driver for x86_64  580.82.07  Release Build  (builder@28e54e79972f)  Thu Apr 30 18:47:14 UTC 2026
[   16.845848] LoadPin: kernel-module pinning-excluded obj="/root/lib/modules/6.6.122+/kernel/drivers/gpu/drm/drm_kms_helper.ko" pid=1337 cmdline="insmod /root/lib/modules/6.6.122+/kernel/drivers/gpu/drm/drm_kms_helper.ko"
[   16.879642] LoadPin: kernel-module pinning-excluded obj="/usr/local/nvidia/drivers/nvidia-drm.ko" pid=1338 

In [6]:
import subprocess

cmd = [
    "/content/vllm-env/bin/vllm", "serve",
    "/root/.cache/huggingface/hub/models--sanskar003--Qwen3.5-9B-AWQ/snapshots/acf12fac67b9591c99f97f6716a9d0e7c20bbff5",
    "--dtype", "float16",
    "--max-model-len", "2048",
    "--max-num-seqs", "8",
    "--gpu-memory-utilization", "0.80",
    "--host", "0.0.0.0",
    "--port", "8000",
    "--served-model-name", "qwen3.5-9b",
]

with open("/content/vllm_server.log", "w") as log:
    process = subprocess.Popen(
        cmd,
        stdout=log,
        stderr=subprocess.STDOUT,
        start_new_session=True
    )

print("vLLM started.")
print("PID:", process.pid)

vLLM started.
PID: 30843


In [7]:
!tail -20 /content/vllm_server.log

(EngineCore pid=31021) INFO 09-21 04:51:26 [mm_encoder_attention.py:372] Using AttentionBackendEnum.TORCH_SDPA for MMEncoderAttention.
(EngineCore pid=31021) INFO 09-21 04:51:26 [compressed_tensors_wNa16.py:137] Using MarlinLinearKernel for CompressedTensorsWNA16
(EngineCore pid=31021) INFO 09-21 04:51:26 [qwen_gdn_linear_attn.py:167] Using Triton/FLA GDN prefill kernel (requested=auto, head_k_dim=128).
(EngineCore pid=31021) INFO 09-21 04:51:26 [qwen_gdn_linear_attn.py:511] Falling back to the Triton GDN decode path: the fused CUDA kernel requires a BF16 GDN model with K=V=128, SiLU or sigmoid gating, non-interleaved GQA layout, BF16 convolution cache, BF16 or FP32 recurrent state, and a GPU with compute capability 8.0+
(EngineCore pid=31021) INFO 09-21 04:51:26 [qwen_gdn_linear_attn.py:519] GDN decode kernel: triton
(EngineCore pid=31021) INFO 09-21 04:51:27 [cuda.py:492] Using TRITON_ATTN attention backend out of potential backends: ['TRITON_ATTN', 'FLEX_ATTENTION'].
(EngineCore pid

In [9]:
!tail -10 /content/vllm_server.log

(APIServer pid=30843) INFO 09-21 04:55:11 [launcher.py:70] Route: /is_scaling_elastic_ep, Methods: POST
(APIServer pid=30843) INFO 09-21 04:55:11 [launcher.py:70] Route: /v1/chat/completions/render, Methods: POST
(APIServer pid=30843) INFO 09-21 04:55:11 [launcher.py:70] Route: /v1/messages/render, Methods: POST
(APIServer pid=30843) INFO 09-21 04:55:11 [launcher.py:70] Route: /v1/completions/render, Methods: POST
(APIServer pid=30843) INFO 09-21 04:55:11 [launcher.py:70] Route: /v1/chat/completions/derender, Methods: POST
(APIServer pid=30843) INFO 09-21 04:55:11 [launcher.py:70] Route: /v1/completions/derender, Methods: POST
(APIServer pid=30843) INFO 09-21 04:55:11 [launcher.py:70] Route: /inference/v1/generate, Methods: POST
(APIServer pid=30843) INFO:     Started server process [30843]
(APIServer pid=30843) INFO:     Waiting for application startup.
(APIServer pid=30843) INFO:     Application startup complete.


In [10]:
import requests

r = requests.get("http://127.0.0.1:8000/v1/models", timeout=10)

print("Status:", r.status_code)
print(r.json())

Status: 200
{'object': 'list', 'data': [{'id': 'qwen3.5-9b', 'object': 'model', 'created': 1789966820, 'owned_by': 'vllm', 'root': '/root/.cache/huggingface/hub/models--sanskar003--Qwen3.5-9B-AWQ/snapshots/acf12fac67b9591c99f97f6716a9d0e7c20bbff5', 'parent': None, 'max_model_len': 2048, 'permission': [{'id': 'modelperm-a9fa74424e7d43ea', 'object': 'model_permission', 'created': 1789966820, 'allow_create_engine': False, 'allow_sampling': True, 'allow_logprobs': True, 'allow_search_indices': False, 'allow_view': True, 'allow_fine_tuning': False, 'organization': '*', 'group': None, 'is_blocking': False}]}]}


In [11]:
from openai import OpenAI

client = OpenAI(
    base_url="http://127.0.0.1:8000/v1",
    api_key="dummy"
)

response = client.chat.completions.create(
    model="qwen3.5-9b",
    messages=[
        {
            "role": "user",
            "content": "In one sentence, explain what a knowledge graph is."
        }
    ],
    max_tokens=100,
    temperature=0.2,
)

print(response.choices[0].message.content)

Thinking Process:

1.  **Analyze the Request:**
    *   Task: Explain what a knowledge graph is.
    *   Constraint: In one sentence.

2.  **Define "Knowledge Graph":**
    *   What is it? A structured representation of information.
    *   What does it do? Connects entities (people, places, things) and their relationships.
    *   What is the goal? To make data understandable and searchable for
